# PRUEBAS


In [23]:
import pandas as pd
import time
from datetime import datetime

# Cargar configuración DINAMIDA de acuerdo al entorno
from dotenv import dotenv_values
import os
import sys

# Acceso a Datos
import psycopg2 as pg2
import pyodbc
import sqlalchemy
from sqlalchemy import text

# Determinar la ruta base del proyecto
# BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
BASE_DIR = "C:/ETL/FORECAST"  # Ruta fija para desarrollo local
CORE_DIR = os.path.join(BASE_DIR, 'forecast_core')
sys.path.insert(0, CORE_DIR)
print("Contenido de sys.path:")
for path in sys.path:
    print(path)

ENV_PATH = os.environ.get("FORECAST_ENV_PATH", "C:/ETL/FORECAST/.env")  # Toma Producción si está definido, o la ruta por defecto
if not os.path.exists(ENV_PATH):
    print(f"El archivo .env no existe en la ruta: {ENV_PATH}")
    print(f"Directorio actual: {os.getcwd()}")
    sys.exit(1)
    
secrets = dotenv_values(ENV_PATH)
folder = f"{secrets['BASE_DIR']}/{secrets['FOLDER_DATOS']}"

# Verificar en que entorno está funcioando
print(f"Python executable: {sys.executable}")
print(f"PATH: {os.environ.get('PATH')}")


Contenido de sys.path:
C:/ETL/FORECAST\forecast_core
C:/ETL/FORECAST\forecast_core
C:/ETL/FORECAST\forecast_core
C:/ETL/FORECAST\forecast_core
C:\Program Files\Python312\python312.zip
C:\Program Files\Python312\DLLs
C:\Program Files\Python312\Lib
C:\Program Files\Python312
c:\ETL\venv

c:\ETL\venv\Lib\site-packages
c:\ETL\venv\Lib\site-packages\win32
c:\ETL\venv\Lib\site-packages\win32\lib
c:\ETL\venv\Lib\site-packages\Pythonwin
Python executable: c:\ETL\venv\Scripts\python.exe
PATH: c:\ETL\venv\Scripts;C:\ETL\venv\Scripts;C:\Program Files\Python312\Scripts\;C:\Program Files\Python312\;C:\windows\system32;C:\windows;C:\windows\System32\Wbem;C:\windows\System32\WindowsPowerShell\v1.0\;C:\windows\System32\OpenSSH\;C:\Program Files\Git\cmd;C:\Program Files\HP\HP One Agent;C:\Program Files\Microsoft SQL Server\150\Tools\Binn\;C:\Program Files\Microsoft SQL Server\Client SDK\ODBC\170\Tools\Binn\;C:\Program Files (x86)\Microsoft SQL Server\160\DTS\Binn\;C:\WINDOWS\system32;C:\WINDOWS;C:\WIND

In [21]:
# Solo importa lo necesario desde el módulo de funciones
# from forecast_core.funciones_forecast import (
#     # get_forecast,
#     generar_datos,
#     Procesar_ALGO_01,
#     Procesar_ALGO_02,
#     Procesar_ALGO_03,
#     Procesar_ALGO_04,
#     Procesar_ALGO_05,
#     Procesar_ALGO_06,
#     Procesar_ALGO_07,     
#     generar_datos,    
#     get_execution_execute_by_status,
#     get_full_parameters,
#     update_execution_execute
# )


def Open_Connexa_Alquemy():
    DB_TYPE = "postgresql"
    DB_USER = secrets['PGP_USER']
    DB_PASS = secrets['PGP_PASSWORD']  # ⚠️ Reemplazar por la contraseña real o usar variables de entorno
    DB_HOST = secrets['PGP_HOST']
    DB_PORT = secrets['PGP_PORT']
    DB_NAME = secrets['PGP_DB']

    # Crear engine de conexión
    try:
        engine = sqlalchemy.create_engine(
        f"{DB_TYPE}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
        )
        conn = engine.connect()
        return conn
    except Exception as e:
        print(f'Error en la conexión: {e}')
        return None   

def Close_Connection(conn): 
    if conn is not None:
        conn.close()
        # print("✅ Conexión cerrada.")    
    return True

def Open_Conn_Postgres():
    conn_str = f"dbname={secrets['PGP_DB']} user={secrets['PGP_USER']} password={secrets['PGP_PASSWORD']} host={secrets['PGP_HOST']} port={secrets['PGP_PORT']}"
    for i in range(5):
        try:
            conn = pg2.connect(conn_str)
            return conn 
        except Exception as e:
            print(f"Error en la conexión, intento {i+1}/{5}: {e}")
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

In [ ]:
def extender_datos_forecast(algoritmo, name, id_proveedor):
    # Recuperar Historial de Ventas
    df_ventas = pd.read_csv(f'{folder}/{name}_Ventas.csv')

    # Convertir tipos de datos    
    df_ventas['Codigo_Articulo'] = pd.to_numeric(df_ventas['Codigo_Articulo'], errors='coerce').astype('Int64')
    df_ventas['Sucursal'] = pd.to_numeric(df_ventas['Sucursal'], errors='coerce').astype('Int64')   
    df_ventas['Fecha']= pd.to_datetime(df_ventas['Fecha'])

    if df_ventas[['Codigo_Articulo', 'Sucursal']].isnull().any().any():
        print(f"⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.")
        df_ventas = df_ventas.dropna(subset=['Codigo_Articulo', 'Sucursal'], how='all')  # Eliminar filas donde ambas columnas son NaN
        print(f"⚠️ Atención: Se ELIMINARON registros de Ventas que contiene registros con valores nulos en Código o Sucursal.")
    
    # Recuperar Maestro de Artículos
    articulos = pd.read_csv(f'{folder}/{name}_articulos.csv')
    
    # Recuperar Maestro de Artículos
    stock_sucursal = pd.read_csv(f'{folder}/{name}_stock_sucursal.csv')

    # Recuperando Forecast Calculado
    df_forecast = pd.read_csv(f'{folder}/{algoritmo}_Solicitudes_Compra.csv')
    df_forecast.fillna(0)   # Por si se filtró algún missing value
    print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")
    
    conn = Open_Conn_Postgres()
    
    # Obtener Sites
    stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
    stores = stores[pd.to_numeric(stores['code'], errors='coerce').notna()].copy()
    stores['code'] = stores['code'].astype(int)

    # Obtener Productos
    products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore
    products = products[pd.to_numeric(products['ext_code'], errors='coerce').notna()].copy()
    products['ext_code'] = products['ext_code'].astype(int)
    assert products['ext_code'].is_unique, "ERROR: productos.ext_code tiene duplicados"


    Close_Connection(conn)

    # Unir con productos y validar
    df_merged = df_forecast.merge(products, left_on='Codigo_Articulo', right_on='ext_code', how='left')
    df_merged.rename(columns={'id': 'product_id'}, inplace=True)
    df_merged.drop(columns=['ext_code', 'description'], inplace=True)

    # Unir con sites y validar
    df_merged = df_merged.merge(stores, left_on='Sucursal', right_on='code', how='left')
    df_merged.rename(columns={'id': 'site_id'}, inplace=True)
    df_merged.drop(columns=['code', 'name'], inplace=True)

    # Validación de integridad referencial
    errores = df_merged[df_merged['site_id'].isna() | df_merged['product_id'].isna()]
    if not errores.empty:
        print(f"❌ Error: Se encontraron {len(errores)} registros con site_id o product_id nulos.")
        errores[['Codigo_Articulo', 'Sucursal']].drop_duplicates().to_csv(
            f"{folder}/{algoritmo}_Errores_Missing_UUID.csv", index=False)
        raise ValueError("Existen artículos o sucursales no presentes en Connexa. Revisión necesaria.")

    
    df_nuevo = articulos.copy()   # Articulos ya tiene SP_BASE_ARTICULOS_SUCURSAL
    # -- COMBINAR ARTÍCULOS y STOCK --
    df_nuevo = pd.merge(df_nuevo, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')
    df_nuevo.drop(columns=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], inplace=True) 
    df_nuevo['C_SUCU_EMPR'] = df_nuevo['C_SUCU_EMPR'].astype(int)
    df_nuevo['C_ARTICULO'] = df_nuevo['C_ARTICULO'].astype(int)

    duplicados = df_nuevo[df_nuevo.duplicated(['C_SUCU_EMPR', 'C_ARTICULO'], keep=False)]
    if not duplicados.empty:
        print(f"🚨 Hay {len(duplicados)} filas duplicadas en df_nuevo por artículo+sucursal")
        duplicados.to_csv(f"{folder}/Duplicados_df_nuevo.csv", index=False)

        #Elimino Duplicados
        df_nuevo = df_nuevo.drop_duplicates(subset=['C_SUCU_EMPR', 'C_ARTICULO'])

    df_merged = df_merged.merge(
        df_nuevo,
        left_on=['Sucursal', 'Codigo_Articulo'],
        right_on=['C_SUCU_EMPR', 'C_ARTICULO'],
        how='left'
    )
    df_merged.drop(columns=['C_SUCU_EMPR', 'C_ARTICULO'], inplace=True)

    print(f"🔍 Forecast original: {len(df_forecast)} registros")
    print(f"📈 Forecast extendido: {len(df_merged)} registros")

    return df_merged

In [30]:
name = "34229_Resistack"
algoritmo = "ALGO_05"
id_proveedor = 34229

In [31]:
print(f'{folder}/{name}_{algoritmo}_Solicitudes_Compra.csv')

C:/ETL/FORECAST/data/34229_Resistack_ALGO_05_Solicitudes_Compra.csv


In [6]:
# Recuperar Historial de Ventas
df_ventas = pd.read_csv(f'{folder}/{name}_Ventas.csv')

# Convertir tipos de datos    
df_ventas['Codigo_Articulo'] = pd.to_numeric(df_ventas['Codigo_Articulo'], errors='coerce').astype('Int64')
df_ventas['Sucursal'] = pd.to_numeric(df_ventas['Sucursal'], errors='coerce').astype('Int64')   
df_ventas['Fecha']= pd.to_datetime(df_ventas['Fecha'])

if df_ventas[['Codigo_Articulo', 'Sucursal']].isnull().any().any():
    print(f"⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.")
    df_ventas = df_ventas.dropna(subset=['Codigo_Articulo', 'Sucursal'], how='all')  # Eliminar filas donde ambas columnas son NaN
    print(f"⚠️ Atención: Se ELIMINARON registros de Ventas que contiene registros con valores nulos en Código o Sucursal.")

⚠️ Atención: Ventas contiene registros con valores nulos en Código o Sucursal.
⚠️ Atención: Se ELIMINARON registros de Ventas que contiene registros con valores nulos en Código o Sucursal.


In [ ]:
# Recuperar Maestro de Artículos
articulos = pd.read_csv(f'{folder}/{name}_articulos.csv')

# Recuperar Maestro de Artículos
stock_sucursal = pd.read_csv(f'{folder}/{name}_stock_sucursal.csv')

# Recuperando Forecast Calculado
df_forecast = pd.read_csv(f'{folder}/{name}_{algoritmo}_Solicitudes_Compra.csv')
df_forecast.fillna(0)   # Por si se filtró algún missing value
print(f"-> Datos Recuperados del CACHE: {id_proveedor}, Label: {name}")

-> Datos Recuperados del CACHE: 34229, Label: 34229_Resistack


In [24]:
# Completar con UUIDs de Connexa
conn = Open_Conn_Postgres()
    
# Obtener Sites
stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
stores = stores[pd.to_numeric(stores['code'], errors='coerce').notna()].copy()
stores['code'] = stores['code'].astype(int)

# Obtener Productos
products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore
products = products[pd.to_numeric(products['ext_code'], errors='coerce').notna()].copy()
products['ext_code'] = products['ext_code'].astype(int)
assert products['ext_code'].is_unique, "ERROR: productos.ext_code tiene duplicados"


Close_Connection(conn)

C:\Users\eduar\AppData\Local\Temp\ipykernel_37520\3847066814.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stores = pd.read_sql("SELECT code, name, id FROM public.fnd_site", conn) # type: ignore
C:\Users\eduar\AppData\Local\Temp\ipykernel_37520\3847066814.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT ext_code, description, id FROM public.fnd_product", conn) # type: ignore


True

In [25]:
# Unir con productos y validar
df_merged = df_forecast.merge(products, left_on='Codigo_Articulo', right_on='ext_code', how='left')
df_merged.rename(columns={'id': 'product_id'}, inplace=True)
df_merged.drop(columns=['ext_code', 'description'], inplace=True)


In [26]:
# Unir con sites y validar
df_merged = df_merged.merge(stores, left_on='Sucursal', right_on='code', how='left')
df_merged.rename(columns={'id': 'site_id'}, inplace=True)
df_merged.drop(columns=['code', 'name'], inplace=True)

In [27]:
# Validación de integridad referencial
errores = df_merged[df_merged['site_id'].isna() | df_merged['product_id'].isna()]
if not errores.empty:
    print(f"❌ Error: Se encontraron {len(errores)} registros con site_id o product_id nulos.")
    errores[['Codigo_Articulo', 'Sucursal']].drop_duplicates().to_csv(
        f"{folder}/{algoritmo}_Errores_Missing_UUID.csv", index=False)
    raise ValueError("Existen artículos o sucursales no presentes en Connexa. Revisión necesaria.")

In [29]:
df_nuevo = articulos.copy()   # Articulos ya tiene SP_BASE_ARTICULOS_SUCURSAL
# -- COMBINAR ARTÍCULOS y STOCK --
df_nuevo = pd.merge(df_nuevo, stock_sucursal, left_on=['C_ARTICULO', 'C_SUCU_EMPR'], right_on=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], how='inner')
df_nuevo.drop(columns=['CODIGO_ARTICULO', 'CODIGO_SUCURSAL'], inplace=True) 
df_nuevo['C_SUCU_EMPR'] = df_nuevo['C_SUCU_EMPR'].astype(int)
df_nuevo['C_ARTICULO'] = df_nuevo['C_ARTICULO'].astype(int)

In [32]:
duplicados = df_nuevo[df_nuevo.duplicated(['C_SUCU_EMPR', 'C_ARTICULO'], keep=False)]
if not duplicados.empty:
    print(f"🚨 Hay {len(duplicados)} filas duplicadas en df_nuevo por artículo+sucursal")
    duplicados.to_csv(f"{folder}/Duplicados_df_nuevo.csv", index=False)

    #Elimino Duplicados
    df_nuevo = df_nuevo.drop_duplicates(subset=['C_SUCU_EMPR', 'C_ARTICULO'])

df_merged = df_merged.merge(
    df_nuevo,
    left_on=['Sucursal', 'Codigo_Articulo'],
    right_on=['C_SUCU_EMPR', 'C_ARTICULO'],
    how='left'
)
df_merged.drop(columns=['C_SUCU_EMPR', 'C_ARTICULO'], inplace=True)

print(f"🔍 Forecast original: {len(df_forecast)} registros")
print(f"📈 Forecast extendido: {len(df_merged)} registros")


🔍 Forecast original: 106 registros
📈 Forecast extendido: 106 registros
